# Extraction

Outra aplicação comum para funções é a realização de extração de conteúdos do texto informado. Isso facilita o parseamento de informações em grande escala, em que podemos utilizar scripts para limpar textos informados, mantendo apenas o necessário da informação que o usuário necessita. Vamos começar com um exemplo simples, extraindo a informação de datas e acontecimentos de um texto.

## Extraindo datas e acontecimentos

Digamos que temos um texto como a seguir e queremos extrair datas e acontecimentos que aparecem no texto:

In [2]:
texto = """
A Apple foi fundada em 1 de abril de 1976 por Steve Wozniak, Steve Jobs e Ronald Wayne 
com o nome de Apple Computers, na Califórnia. O nome foi escolhido por Jobs após a visita do pomar 
de maçãs da fazenda de Robert Friedland, também pelo fato do nome soar bem e ficar antes da Atari 
nas listas telefônicas.

O primeiro protótipo da empresa foi o Apple I que foi demonstrado na Homebrew Computer Club em 1975, 
as vendas começaram em julho de 1976 com o preço de US$ 666,66, aproximadamente 200 unidades foram 
vendidas,[21] em 1977 a empresa conseguiu o aporte de Mike Markkula e um empréstimo do Bank of America.
"""

In [4]:
from pydantic import BaseModel, Field
from typing import List
from langchain_core.utils.function_calling import convert_to_openai_function

class Acontecimento(BaseModel):
	"""Informação sobre um acontecimento"""
	data: str = Field(description="Data do acontecimento no formato YYYY-MM-DD")
	acontecimento: str = Field(description="Acontecimento extraído do texto")

class ListaAcontecimentos(BaseModel):
	"""Acontecimentos para extração"""
	acontecimentos: List[Acontecimento] = Field(description="Lista de acontecimentos presentes no texto informado")

tool_acontecimentos = convert_to_openai_function(ListaAcontecimentos)
tool_acontecimentos

{'name': 'ListaAcontecimentos',
 'description': 'Acontecimentos para extração',
 'parameters': {'properties': {'acontecimentos': {'description': 'Lista de acontecimentos presentes no texto informado',
    'items': {'description': 'Informação sobre um acontecimento',
     'properties': {'data': {'description': 'Data do acontecimento no formato YYYY-MM-DD',
       'type': 'string'},
      'acontecimento': {'description': 'Acontecimento extraído do texto',
       'type': 'string'}},
     'required': ['data', 'acontecimento'],
     'type': 'object'},
    'type': 'array'}},
  'required': ['acontecimentos'],
  'type': 'object'}}

In [5]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
	("system", "Extraia as frases de acontecimentos. Elas devem ser extraídas integralmente"),
	("user", "{input}")
])

chat = ChatOpenAI()

chain = (prompt 
		 | chat.bind(functions=[tool_acontecimentos], function_call={"name": 'ListaAcontecimentos'}))

In [6]:
chain.invoke({"input": texto})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"acontecimentos":[{"data":"1976-04-01","acontecimento":"A Apple foi fundada por Steve Wozniak, Steve Jobs e Ronald Wayne na Califórnia."},{"data":"1975","acontecimento":"O Apple I foi demonstrado na Homebrew Computer Club."},{"data":"1976-07","acontecimento":"Início das vendas do Apple I por US$ 666,66."},{"data":"1977","acontecimento":"A empresa recebeu o aporte de Mike Markkula e um empréstimo do Bank of America."}]}', 'name': 'ListaAcontecimentos'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 325, 'total_tokens': 450, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--08c274d8-482e-4c66-aa85-

In [7]:
from langchain.output_parsers.openai_functions import JsonOutputFunctionsParser

chain = (prompt 
         | chat.bind(functions=[tool_acontecimentos], function_call={'name': 'ListaAcontecimentos'})
         | JsonOutputFunctionsParser())

chain.invoke({'input': texto})

{'acontecimentos': [{'data': '1976-04-01',
   'acontecimento': 'A Apple foi fundada por Steve Wozniak, Steve Jobs e Ronald Wayne na Califórnia com o nome de Apple Computers.'},
  {'data': '1975',
   'acontecimento': 'O primeiro protótipo da empresa, o Apple I, foi demonstrado na Homebrew Computer Club.'},
  {'data': '1976-07',
   'acontecimento': 'As vendas do Apple I começaram com o preço de US$ 666,66. Aproximadamente 200 unidades foram vendidas.'},
  {'data': '1977',
   'acontecimento': 'A empresa conseguiu o aporte de Mike Markkula e um empréstimo do Bank of America.'}]}

In [8]:
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser

chain = (prompt 
         | chat.bind(functions=[tool_acontecimentos], function_call={'name': 'ListaAcontecimentos'})
         | JsonKeyOutputFunctionsParser(key_name='acontecimentos'))

chain.invoke({'input': texto})

[{'data': '1976-04-01',
  'acontecimento': 'Apple foi fundada por Steve Wozniak, Steve Jobs e Ronald Wayne na Califórnia com o nome de Apple Computers.'},
 {'data': '1975-07',
  'acontecimento': 'Vendas do Apple I começaram em julho de 1976 com o preço de US$ 666,66.'},
 {'data': '1977',
  'acontecimento': 'Apple conseguiu o aporte de Mike Markkula e um empréstimo do Bank of America.'}]

## Extraindo informações da web

A aplicação de extração pode ser muito utilizada quando combinada com técnicas de WebScraping. Em geral, em WebScraping estamos buscando informações em páginas web. Em sua grande maioria, essas informações virão completamente desformatadas e em html, o que dificulta a utilização da informação. Podemos criar aplicações utilizando as técnicas que aprendemos para conseguir as informações específicas que precisamos das páginas que estamos analisando.

Vamos dar um exemplo analisando a página de blog da Asimov e tentando extrair todos os posts contidos na página.
https://hub.asimov.academy/blog/

In [9]:
from langchain_community.document_loaders.web_base import WebBaseLoader

loader = WebBaseLoader("https://hub.asimov.academy/blog/")
page = loader.load()
page

[Document(metadata={'source': 'https://hub.asimov.academy/blog/', 'title': 'Blog de Python, IA e Data Science | Asimov Academy', 'description': 'Confira os principais assuntos de programação em Python, Inteligência Artificial e Data Science no blog da Asimov Academy!', 'language': 'pt-BR'}, page_content='\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nBlog de Python, IA e Data Science | Asimov Academy\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nTamanho de fonte\n\nA\nA\nA\nA\n\n\n\nAlto contraste\n\nLigado\nDesligado\n\n\n\nAltura de linha\n\n1\n1.3\n1.5\n1.7\n2.0\n\n\n\n \n\n\n\nPesquisar na plataforma\n\n\n\n\n\nEntrar\nCadastrar\n\n\n\n\n\nBlog\n\n\nArtigos\n\n\nTutoriais\n\n\nMateriais Extras\n\n\nEscreva para o blog!\n200xp \n\n\n\n\nBlog de Python, IA e Data Science da Asimov\n\n        Artigos completos sobre Python, Ciência de Dados e Inteligência Artificial. Fique por dentr

In [10]:
from pydantic import BaseModel, Field
from typing import List
from langchain_core.utils.function_calling import convert_to_openai_function

class BlogPost(BaseModel):
	"""Informações sobre um post do blog"""
	titulo: str = Field(description="Título do post do blog")
	autor: str = Field(description="Autor do post do blog")
	data: str = Field(description="Data de publicação do post do blog")

class BlogSite(BaseModel):
	"""Lista de blog posts de um site"""
	posts: List[BlogPost] = Field(description="Lista de posts de blog do site")

tool_blog = convert_to_openai_function(BlogSite)
tool_blog

{'name': 'BlogSite',
 'description': 'Lista de blog posts de um site',
 'parameters': {'properties': {'posts': {'description': 'Lista de posts de blog do site',
    'items': {'description': 'Informações sobre um post do blog',
     'properties': {'titulo': {'description': 'Título do post do blog',
       'type': 'string'},
      'autor': {'description': 'Autor do post do blog', 'type': 'string'},
      'data': {'description': 'Data de publicação do post do blog',
       'type': 'string'}},
     'required': ['titulo', 'autor', 'data'],
     'type': 'object'},
    'type': 'array'}},
  'required': ['posts'],
  'type': 'object'}}

In [13]:
from langchain.output_parsers.openai_functions import JsonKeyOutputFunctionsParser
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
	("system", "Extraia da página todos os posts de blog com autor e data de publicação"),
	("user", "{input}")
])

chat = ChatOpenAI()

chain = (prompt 
		 | chat.bind(functions=[tool_blog], function_call={"name": 'BlogSite'})
		 | JsonKeyOutputFunctionsParser(key_name='posts'))

In [15]:
chain.invoke({"input": page})

[{'titulo': 'Asimov Skills: o que \\xe9, como acessar e benef\\xedcios para alunos',
  'autor': 'Carolina Carvalho',
  'data': '12 horas atr&#225;s'},
 {'titulo': 'Bolt.new: o que \\xe9, como funciona e como criar sites e apps com IA ',
  'autor': 'Rebeca Hon&#243;rio',
  'data': '4 dias atr&#225;s'},
 {'titulo': 'Forma&#231;&#227;o Engenheiro de Agentes de IA: tudo o que voc&#234; precisa saber ',
  'autor': 'Carolina Carvalho',
  'data': '12 horas atr&#225;s'},
 {'titulo': 'Base44: o que \\xe9, como funciona e como criar aplicativos com IA em poucos minutos',
  'autor': 'Carolina Carvalho',
  'data': '4 dias atr&#225;s'},
 {'titulo': 'Power Query: guia completo para iniciantes em Excel e Power BI',
  'autor': 'Rebeca Hon&#243;rio',
  'data': '17 dias atr&#225;s'},
 {'titulo': 'O que \\xe9 Tkinter e como criar interfaces gr&#225;ficas em Python',
  'autor': 'Rebeca Hon&#243;rio',
  'data': '20 dias atr&#225;s'},
 {'titulo': 'Modelagem de Dados: o que \\xe9, tipos e como aplicar na pr&